In [ ]:
!pip install transformers[torch] datasets evaluate rouge_score accelerate
!pip install fastapi uvicorn[standard] python-multipart

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.7 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=eee402f2500989a484210986d49d7043e396deeea0f0cfcb1d368269bafd21be
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.8/510.8 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 452.2/452.2 kB 31.9 MB/s eta 0:00:00


In [ ]:
from datasets import load_dataset
dataset = load_dataset("cnn_dailymail", "3.0.0")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/34.7M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

In [ ]:
from transformers import AutoTokenizer
model_checkpoint = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
def preprocess_function(examples):
    # Prefix for T5 models, not strictly needed for BART but good practice
    prefix = "summarize: "
    inputs = [prefix + doc for doc in examples["article"]]

    # Tokenize the articles
    model_inputs = tokenizer(inputs, max_length=1024, truncation=True, padding="max_length")

    # Tokenize the summaries (labels)
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(examples["highlights"], max_length=128, truncation=True, padding="max_length")

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [ ]:
tokenized_datasets = dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/287113 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4007: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/13368 [00:00<?, ? examples/s]

Map:   0%|          | 0/11490 [00:00<?, ? examples/s]

In [ ]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./bart-summarizer",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=3,
    predict_with_generate=True,
)

In [ ]:
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainer, DataCollatorForSeq2Seq

model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"].select(range(1000)), # Use a subset for faster demo
    eval_dataset=tokenized_datasets["validation"].select(range(200)),
    tokenizer=tokenizer,
    data_collator=data_collator,
)

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

/tmp/ipython-input-336792927.py:6: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: idontknow-cpp (idontknow-cpp-nitte-meenakshi-ins) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
500,0.655100


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:4034: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 142, 'min_length': 56, 'early_stopping': True, 'num_beams': 4, 'length_penalty': 2.0, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


TrainOutput(global_step=750, training_loss=0.5190397796630859, metrics={'train_runtime': 1698.0778, 'train_samples_per_second': 1.767, 'train_steps_per_second': 0.442, 'total_flos': 6501313806336000.0, 'train_loss': 0.5190397796630859, 'epoch': 3.0})

In [ ]:
trainer.save_model("./my_fine_tuned_bart_summarizer")
tokenizer.save_pretrained("./my_fine_tuned_bart_summarizer")

('./my_fine_tuned_bart_summarizer/tokenizer_config.json',
 './my_fine_tuned_bart_summarizer/special_tokens_map.json',
 './my_fine_tuned_bart_summarizer/vocab.json',
 './my_fine_tuned_bart_summarizer/merges.txt',
 './my_fine_tuned_bart_summarizer/added_tokens.json',
 './my_fine_tuned_bart_summarizer/tokenizer.json')

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
from transformers import pipeline
import torch

# Initialize the FastAPI app
app = FastAPI(
    title="Text Summarization API",
    description="An API to summarize text using a fine-tuned BART model.",
)

# Load your fine-tuned model using the pipeline for convenience
# This loads the model into memory only once when the app starts
summarizer = pipeline(
    "summarization",
    model="./my_fine_tuned_bart_summarizer",
    tokenizer="./my_fine_tuned_bart_summarizer",
    device=0 if torch.cuda.is_available() else -1 # Use GPU if available
)

# Define the request body model using Pydantic for data validation
class SummarizationRequest(BaseModel):
    text: str
    min_length: int = 50
    max_length: int = 150

# Define the response body model
class SummarizationResponse(BaseModel):
    summary: str

# Define the API endpoint
@app.post("/summarize", response_model=SummarizationResponse)
def summarize_text(request: SummarizationRequest):
    """
    Accepts a long text and returns a summarized version.
    """
    summary_result = summarizer(
        request.text,
        min_length=request.min_length,
        max_length=request.max_length,
        do_sample=False
    )
    return {"summary": summary_result[0]['summary_text']}

@app.get("/")
def read_root():
    return {"message": "Welcome to the Summarization API. Go to /docs for more info."}

/usr/local/lib/python3.12/dist-packages/transformers/models/bart/configuration_bart.py:177: UserWarning: Please make sure the config includes `forced_bos_token_id=0` in future versions. The config can simply be saved and uploaded again to be fixed.
  warnings.warn(
Device set to use cuda:0


In [ ]:
!uvicorn main:app --reload

INFO:     Will watch for changes in these directories: ['/content']
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     Started reloader process [17514] using WatchFiles
ERROR:    Error loading ASGI app. Could not import module "main".
INFO:     Stopping reloader process [17514]


In [ ]:
from google.colab import files

files.download("/content/my_fine_tuned_bart_summarizer/model.safetensors")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>